In [2]:
import json
import ast

# system message
system_msg = 'You are a helpful assistant who is an expert in Kubernetes troubleshooting and can help developers with step by step suggestions about how to investigate and solve Kubernetes problems.'

# File paths
input_file_path = './data/raw_data.txt'
output_file_path = './data/kub_data_mrc.jsonl'

# Read the input file
with open(input_file_path, 'r') as infile:
    lines = infile.readlines()

# Process the file to extract prompt-completion pairs
mrc_dicts = []

for line in lines:
    splitted = line.split('\t```json')
    user_msg = splitted[0]
    # assistant_msg = json.loads(splitted[1])
    # assistant_msg = ast.literal_eval(splitted[1])
    assistant_msg = splitted[1]
    mrc_dicts.append({"messages" : [{"role": "system", "content": system_msg}, 
                         {"role": "user", "content": user_msg}, 
                         {"role": "assistant", "content": assistant_msg}]})    

with open(output_file_path, 'w') as outfile:
    for dct in mrc_dicts:
        json.dump(dct, outfile)
        outfile.write('\n')

mrc_file = output_file_path

In [3]:
from openai import OpenAI
import warnings
import os

In [4]:
os.environ['OPENAI_API_KEY'] = ""
client = OpenAI()

In [5]:
# Upload the file
response_file = client.files.create(
    file=open(mrc_file, "rb"),
    purpose="fine-tune"
)

In [6]:
# Create the fine-tune job
response_fine_tune = client.fine_tuning.jobs.create(
    training_file=response_file.id,
    model='gpt-3.5-turbo-0125',
    suffix='fahim'
)

In [7]:
model_name = 'ft:gpt-3.5-turbo-0125:personal:fahim:9uG09DCK'

def ask_model_mrc(question):
    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": question}
        ]
    )
    return completion.choices[0].message.content

In [8]:
question = "Kubernetes - Failed Persistent Volume Claims"
ask_model_mrc(question)

'{  "investigation": {    "steps": [      {        "description": "Identify the namespace and persistent volume claim name for the failed claim",        "command": "kubectl get pvc -n <namespace>",        "output": "`kubectl get pvc -n my-namespace`\\n```shell\\nNAME            STATUS    VOLUME   CAPACITY   ACCESS MODES   STORAGECLASS   AGE\\nfailed-claim    Failed             10s`\\n```"      },      {        "description": "Describe the persistent volume claim to gather more details",        "command": "kubectl describe pvc <claim-name> -n <namespace>",        "output": "`kubectl describe pvc failed-claim -n my-namespace`\\n```shell\\nName:          failed-claim\\nNamespace:     my-namespace\\n…\\nStatus:        Failed\\n…\\nEvents:\\n  Type     Reason                 Age               From                         Message\\n  ----     ------                 ----              ----                         -------\\n  Warning  ProvisioningFailed     20s (x4 over 35s)  persistentvolume-c